<a href="https://colab.research.google.com/github/abdoraven/hcv-ordinal-classification/blob/main/Ordinal_Classification_of_Hepatitis_C_Severity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ordinal Classification of Hepatitis C Severity (HCV dataset)

Predicts ordinal disease stage (0 = Blood Donor → 4 = Cirrhosis) from routine
blood-test values, using several ordinal-aware models (`mord` Logistic-AT,
plus regressor-based approaches rounded to classes).

**Dataset**: UCI "HCV data" (Hepatitis C Virus for Egyptian patients / blood
donors). Not committed to this repo — download it and place it as
`hcvdat.csv` in the working directory (see the data-loading cell below for
the exact source and columns expected).

---

### Changelog vs. the original notebook (for you to trace/revert changes)

The original notebook had **8 cells**, where cells 1–7 were near-duplicates
of the same pipeline, each testing one tweak (impute strategy, outlier
handling, feature set) by copy-pasting the whole block and re-running it.
That's why results were hard to compare and the file was long.

This version keeps every one of your 7 original experiments — **no logic
was removed or changed** — but restructures the *code* so each experiment is
a small config dict instead of a ~120-line copy/paste block:

1. **`CHANGE 1`** — Setup (imports, metrics, `OutlierCapper`, `LogTransformer`)
   now lives in **one** cell instead of being repeated 7 times.
2. **`CHANGE 2`** — Data loading/target-mapping now lives in **one** cell.
   Per-experiment differences (dropna, feature subset, ratio feature) are
   now handled by the experiment config, not by re-reading the CSV each time.
3. **`CHANGE 3`** — Each of your 7 experiments (original cells 1–7) is now one
   entry in an `EXPERIMENTS` list, tagged with a comment noting which
   original cell it came from, so you can match it back 1:1.
4. **`CHANGE 4`** — A single `run_experiment()` function replaces the 7
   duplicated evaluation loops. The math/metrics (accuracy, MAE, quadratic
   kappa, same CV splits, same `random_state=42`) are untouched.
5. **`CHANGE 5`** — Results are collected into one pandas comparison table at
   the end instead of 7 separate printouts, so you can see all experiments
   side by side. (Purely additive — nothing about scoring changed.)
6. **`CHANGE 6`** — Comments consolidated to English for consistency (some
   original cells mixed English/Arabic). No behavior change.
7. **`CHANGE 7`** — Verbose `pip install` output and old run outputs were
   cleared before committing (see README suggestion below).

If a result looks different from what you remember, the fastest way to
debug is: find the matching `# from original cell N` tag in `EXPERIMENTS`,
and diff its config against the corresponding cell in your original
`Irdm4.ipynb` (kept as `Irdm4_original.ipynb` if you want to keep it around
locally — **don't commit both**, just the clean one, to avoid confusion).

In [1]:
# Install dependencies (unchanged from original cell 0)
!pip install -q mord imbalanced-learn scikit-learn

  Preparing metadata (setup.py) ... done


In [2]:
# CHANGE 1: this cell merges the imports + metric functions + custom
# transformer classes that were previously repeated at the top of every
# original cell (1-7). Nothing about the logic changed, only de-duplicated.

import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import mord  # specialized ordinal classification library
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import cohen_kappa_score, mean_absolute_error
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

# =========================================================================
# Custom Evaluation Metrics for True Ordinal Targets (same as original)
# =========================================================================
def quadratic_kappa_scorer(y_true, y_pred):
    y_pred_clipped = np.clip(np.round(y_pred), 0, 4).astype(int)
    return cohen_kappa_score(y_true, y_pred_clipped, weights="quadratic")


# =========================================================================
# Outlier Capper Transformer (Safe within Pipeline)
# Same class as original cells 1, 2, 5, 6, 7. Original cell 3 used a
# disabled version (active=False, transform returns X unchanged) to test
# "no outlier capping" - that variant is now handled by simply omitting
# this step in the pipeline (see build_pipeline / outlier_step='none'
# below) instead of keeping a second near-duplicate class.
# =========================================================================
class OutlierCapper(BaseEstimator, TransformerMixin):

    def __init__(self, factor=1.5):
        self.factor = factor
        self.lower_bounds_ = {}
        self.upper_bounds_ = {}

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        for col in X_df.columns:
            if col != 1:  # skip Gender column
                Q1 = X_df[col].quantile(0.25)
                Q3 = X_df[col].quantile(0.75)
                IQR = Q3 - Q1
                self.lower_bounds_[col] = Q1 - self.factor * IQR
                self.upper_bounds_[col] = Q3 + self.factor * IQR
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        for col in X_df.columns:
            if col in self.lower_bounds_:
                X_df[col] = np.clip(
                    X_df[col], self.lower_bounds_[col], self.upper_bounds_[col]
                )
        return X_df.to_numpy()


# =========================================================================
# Log Transformer (same as original cell 4)
# =========================================================================
class LogTransformer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        for col in X_df.columns:
            if col != 1:  # skip Gender column (binary categorical)
                X_df[col] = np.log1p(X_df[col])
        return X_df.to_numpy()


MODEL_FACTORY = lambda: {
    "Ordinal Logistic Regression (mord)": mord.LogisticAT(alpha=1.0),
    "Ordinal Random Forest (Reg)": RandomForestRegressor(random_state=42),
    "Ordinal Decision Tree (Reg)": DecisionTreeRegressor(random_state=42),
    "Ordinal KNN (Reg)": KNeighborsRegressor(n_neighbors=5),
}

In [3]:
# CHANGE 2: single data-loading cell (previously re-read + re-mapped in
# every one of the 7 original cells). Loads the raw file and does only the
# steps that were common to ALL 7 experiments (index col, gender encoding,
# ordinal target mapping). Per-experiment differences (dropna, feature
# subset, engineered ratio) are applied later, inside each experiment.
#
# Dataset: UCI HCV dataset. Download it and place it here as "hcvdat.csv"
# (not committed to the repo - see README).
df_raw = pd.read_csv("hcvdat.csv", index_col=0)

# Encode Gender (unchanged)
if "Sex" in df_raw.columns:
    df_raw["Sex"] = df_raw["Sex"].map({"m": 0, "f": 1})

# Explicit ordinal mapping (0 = Healthy -> 4 = Most Severe), unchanged
CAT_MAP = {
    "0=Blood Donor": 0,
    "0s=suspect Blood Donor": 1,
    "1=Hepatitis": 2,
    "2=Fibrosis": 3,
    "3=Cirrhosis": 4,
}
y_raw = df_raw["Category"].map(CAT_MAP)

ALL_12_FEATURES = [
    "Age", "Sex", "ALB", "ALP", "ALT", "AST", "BIL",
    "CHE", "CHOL", "CREA", "GGT", "PROT",
]
CLEAN_9_FEATURES = ["Age", "Sex", "ALP", "ALT", "AST", "CHE", "CHOL", "CREA", "PROT"]

print(f"Loaded {df_raw.shape[0]} rows, {df_raw.shape[1]} columns.")

Loaded 615 rows, 13 columns.


In [4]:
# CHANGE 3 + 4: each of your 7 original cells (1-7) becomes one config
# entry below (tagged with its source cell number), and a single
# run_experiment() function replaces the 7 duplicated evaluation loops.
# The pipeline steps, CV splits (StratifiedKFold, n_splits=5,
# random_state=42), SMOTE settings, and metric calculations are otherwise
# UNCHANGED from the originals.

def prepare_data(dropna, feature_cols, add_ratio_feature):
    """Applies the per-experiment data steps that used to be pasted at the
    top of each original cell."""
    df = df_raw.dropna().copy() if dropna else df_raw.copy()
    y = df["Category"].map(CAT_MAP)

    if add_ratio_feature:
        # from original cell 5: engineered AST/ALT ratio
        df["AST_ALT_Ratio"] = df["AST"] / (df["ALT"] + 0.001)

    cols = feature_cols if feature_cols is not None else ALL_12_FEATURES
    X = df[cols].copy()
    return X, y


def build_pipeline(model, imputer_strategy, outlier_step):
    steps = []
    if imputer_strategy is not None:
        steps.append(("imputer", SimpleImputer(strategy=imputer_strategy)))
    if outlier_step == "cap":
        steps.append(("outlier_capping", OutlierCapper(factor=1.5)))
    elif outlier_step == "log":
        steps.append(("log_transform", LogTransformer()))
    # outlier_step == "none" -> no step added (equivalent to original
    # cell 3's disabled OutlierCapper, just without the dead class)
    steps.append(("scaler", StandardScaler()))
    steps.append(("smote", SMOTE(k_neighbors=1, random_state=42)))
    steps.append(("classifier", model))
    return ImbPipeline(steps)


def run_experiment(exp):
    X, y = prepare_data(
        dropna=exp["dropna"],
        feature_cols=exp["feature_cols"],
        add_ratio_feature=exp.get("add_ratio_feature", False),
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    rows = []
    for name, model in MODEL_FACTORY().items():
        pipeline = build_pipeline(model, exp["imputer_strategy"], exp["outlier_step"])
        accs, maes, kappas = [], [], []
        for train_idx, test_idx in cv.split(X, y):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            pipeline.fit(X_train, y_train)
            preds = pipeline.predict(X_test)
            final_preds = np.clip(np.round(preds), 0, 4).astype(int)
            accs.append(np.mean(final_preds == y_test))
            maes.append(mean_absolute_error(y_test, final_preds))
            kappas.append(cohen_kappa_score(y_test, final_preds, weights="quadratic"))
        rows.append({
            "experiment": exp["name"],
            "model": name,
            "n_features": X.shape[1],
            "accuracy": np.mean(accs),
            "mae": np.mean(maes),
            "quadratic_kappa": np.mean(kappas),
        })
    return rows


EXPERIMENTS = [
    {  # from original cell 1
        "name": "1. Baseline (mean impute, outlier capping, 12 features)",
        "dropna": False,
        "imputer_strategy": "mean",
        "outlier_step": "cap",
        "feature_cols": None,
    },
    {  # from original cell 2
        "name": "2. Drop missing rows (no imputer), outlier capping, 12 features",
        "dropna": True,
        "imputer_strategy": None,
        "outlier_step": "cap",
        "feature_cols": ALL_12_FEATURES,
    },
    {  # from original cell 3
        "name": "3. Mean impute, outlier capping disabled, 12 features",
        "dropna": False,
        "imputer_strategy": "mean",
        "outlier_step": "none",
        "feature_cols": None,
    },
    {  # from original cell 4
        "name": "4. Mean impute, log1p transform instead of capping, 12 features",
        "dropna": False,
        "imputer_strategy": "mean",
        "outlier_step": "log",
        "feature_cols": None,
    },
    {  # from original cell 5
        "name": "5. Mean impute, outlier capping, AST/ALT ratio feature (11 features)",
        "dropna": False,
        "imputer_strategy": "mean",
        "outlier_step": "cap",
        "feature_cols": [c for c in ALL_12_FEATURES if c not in ("AST", "ALT")] + ["AST_ALT_Ratio"],
        "add_ratio_feature": True,
    },
    {  # from original cell 6
        "name": "6. Median impute, outlier capping, 12 features",
        "dropna": False,
        "imputer_strategy": "median",
        "outlier_step": "cap",
        "feature_cols": None,
    },
    {  # from original cell 7
        "name": "7. Median impute, outlier capping, reduced 9 features (drop ALB, GGT, BIL)",
        "dropna": False,
        "imputer_strategy": "median",
        "outlier_step": "cap",
        "feature_cols": CLEAN_9_FEATURES,
    },
]

In [5]:
# CHANGE 5: run all 7 experiments and collect results into one comparison
# table instead of 7 separate printouts. Purely presentational - the
# underlying numbers for each (experiment, model) pair are computed exactly
# as in the corresponding original cell.
all_rows = []
for exp in EXPERIMENTS:
    all_rows.extend(run_experiment(exp))

results_df = pd.DataFrame(all_rows)
results_df["accuracy"] = results_df["accuracy"].round(2)
results_df["mae"] = results_df["mae"].round(2)
results_df["quadratic_kappa"] = results_df["quadratic_kappa"].round(2)

pd.set_option("display.max_colwidth", None)
results_df.sort_values(["experiment", "model"]).reset_index(drop=True)

,experiment,model,n_features,accuracy,mae,quadratic_kappa
0,"1. Baseline (mean impute, outlier capping, 12 features)",Ordinal Decision Tree (Reg),12,0.90,0.17,0.83
1,"1. Baseline (mean impute, outlier capping, 12 features)",Ordinal KNN (Reg),12,0.90,0.15,0.88
2,"1. Baseline (mean impute, outlier capping, 12 features)",Ordinal Logistic Regression (mord),12,0.65,0.43,0.74
3,"1. Baseline (mean impute, outlier capping, 12 features)",Ordinal Random Forest (Reg),12,0.86,0.17,0.89
4,"2. Drop missing rows (no imputer), outlier capping, 12 features",Ordinal Decision Tree (Reg),12,0.90,0.15,0.82
5,"2. Drop missing rows (no imputer), outlier capping, 12 features",Ordinal KNN (Reg),12,0.92,0.11,0.90
6,"2. Drop missing rows (no imputer), outlier capping, 12 features",Ordinal Logistic Regression (mord),12,0.71,0.31,0.81
7,"2. Drop missing rows (no imputer), outlier capping, 12 features",Ordinal Random Forest (Reg),12,0.89,0.15,0.87
8,"3. Mean impute, outlier capping disabled, 12 features",Ordinal Decision Tree (Reg),12,0.89,0.19,0.81
9,"3. Mean impute, outlier capping disabled, 12 features",Ordinal KNN (Reg),12,0.87,0.22,0.78
